# Kv3.1 e Kv3.3 nel modello MNTB – esperimenti di Choudhury et al. 2020 (REF3)

**Obiettivo:** verificare come si comporta il modello `mntb_neuron_KV3.1_KV3.3_prova.nestml`
(Kv3 diviso in Kv3.1 e Kv3.3, prima prova 50/50) nei tre genotipi di Choudhury 2020
e con il blocco di Kv3 da TEA.

| Genotipo / condizione | Nel modello |
|---|---|
| WT | parametri di default del file `.nestml` |
| Kv3.1 KO | `g_Kv3_1` = 0 |
| Kv3.3 KO | `g_Kv3_3` = 0 |
| TEA 1 mM | `g_Kv3_1` = `g_Kv3_3` = 0 (Kv3 bloccato) |

| Esperimento | Choudhury 2020 | Protocollo | Cosa si misura |
|---|---|---|---|
| C1 | Fig. 6 (e testo) | gradino di corrente di 200 ms, si analizza il **primo** spike | half-width, ampiezza, soglia, AHP, correnti durante lo spike |
| C2 | Fig. 1 e 7 | voltage clamp: HP −60, prepulse −90 mV (200 ms) e −40 mV (20 ms), gradini −90…+40 mV | curva I/V, corrente a +10 mV, effetto di TEA |
| C3 | Fig. 8 | voltage clamp da HP −60 e da HP −40 mV | inattivazione stazionaria di Kv3 |
| C4 | Fig. 10 | treni di impulsi da 0.3 ms per 500 ms a 100–400 Hz, ampiezza 0.5–3.5 nA | numero di spike evocati (curve stimolo-risposta) |

**Nota:** nessun parametro del modello è scritto a mano qui: tutto viene letto dal file `.nestml`
compilato nella prima cella. I genotipi cambiano solo `g_Kv3_1` e `g_Kv3_3`.

Dati di confronto (MNTB, topi P14–P25, C ≈ 11 pF, quindi un neurone di taglia simile al modello da 10 pF):
- **half-width**: WT 0.31 ± 0.08 ms, Kv3.1 KO 0.43 ± 0.09 ms, Kv3.3 KO 0.44 ± 0.09 ms;
- **corrente a +10 mV**: WT 11.9 ± 1.2 nA, Kv3.1 KO 9.8 ± 2.8 nA, Kv3.3 KO 10.1 ± 3.0 nA;
  con TEA 6.1 ± 1.4 nA (Kv3.1 KO) e 5.6 ± 2.0 nA (Kv3.3 KO);
- **treni**: nell'MNTB tutti i genotipi seguono fino a 400 Hz (un solo neurone WT fallisce a 400 Hz).

In [ ]:
import os
os.environ["PYNEST_QUIET"] = "1"
import numpy as np
import matplotlib.pyplot as plt
import nest
nest.set_verbosity("M_ERROR")
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

# Si passa il NOME del file: NESTML lo legge soltanto, senza riscriverlo.
MODEL_FILE = "mntb_neuron_KV3.1_KV3.3_prova.nestml"
split_module, split_model = NESTCodeGeneratorUtils.generate_code_for(MODEL_FILE)

dt = 0.01   # ms, passo di integrazione

nest.ResetKernel()
nest.Install(split_module)
DEFAULTS = nest.GetDefaults(split_model)
print(f"Modello: {split_model}")
print(f"g_Kv3_1 = {DEFAULTS['g_Kv3_1']:g} nS, g_Kv3_3 = {DEFAULTS['g_Kv3_3']:g} nS, "
      f"g_LT = {DEFAULTS['g_LT']:g} nS, g_L = {DEFAULTS['g_L']:g} nS, g_Na = {DEFAULTS['g_Na']:g} nS, C_m = {DEFAULTS['C_m']:g} pF")

# Genotipi: solo le conduttanze Kv3 cambiano, tutto il resto resta come nel modello
GENOTYPES = {
    "WT": {},
    "Kv3.1 KO": {"g_Kv3_1": 0.0},
    "Kv3.3 KO": {"g_Kv3_3": 0.0},
    "TEA (Kv3 bloccato)": {"g_Kv3_1": 0.0, "g_Kv3_3": 0.0},
}
COLORS = {"WT": "tab:blue", "Kv3.1 KO": "tab:green", "Kv3.3 KO": "tab:red", "TEA (Kv3 bloccato)": "k"}
CURRENTS = ["I_Na", "I_LT", "I_Kv3_1", "I_Kv3_3", "I_L"]

summary = {}   # riepilogo finale: misura -> (valore modello, valore paper, note)


def simulate(params, amp_times, amp_values, T, record=("V_m",)):
    # Simula il neurone con un generatore di corrente a gradini.
    # Ritardo della connessione = dt (il default di NEST e' 1 ms): lo stimolo arriva quando lo chiediamo.
    nest.ResetKernel()
    nest.SetKernelStatus({"resolution": dt})
    nest.Install(split_module)
    neuron = nest.Create(split_model, params=params)
    stim = nest.Create("step_current_generator",
                       params={"amplitude_times": amp_times, "amplitude_values": amp_values})
    mm = nest.Create("multimeter", params={"record_from": list(record), "interval": dt})
    nest.Connect(stim, neuron, syn_spec={"delay": dt})
    nest.Connect(mm, neuron)
    nest.Simulate(T)
    ev = mm.get("events")
    order = np.argsort(ev["times"])
    return ev["times"][order], {k: ev[k][order] for k in record}

## C1 – Primo spike evocato da un gradino di 200 ms (Choudhury Fig. 6)

Choudhury porta la membrana a circa −60 mV e inietta un gradino di corrente di 200 ms; analizza il
**primo** spike. Qui il modello parte dal suo riposo e il gradino ha ampiezza `STEP_AMP`.

Misure sul primo spike:
- **ampiezza** = picco − potenziale prima dello stimolo;
- **soglia** = tensione a cui dV/dt supera `THR_DVDT` (50 mV/ms, vedi commento nel codice);
- **half-width** = durata a metà ampiezza, cioè a metà tra il potenziale prima dello stimolo e il picco (come negli altri notebook);
- **ampiezza da soglia** = picco − soglia;
- **tempo di salita 10–90%** e **AHP** (minimo nei 5 ms dopo il picco − soglia: durante il gradino
  il potenziale non torna al riposo, quindi la soglia è un riferimento migliore);
- **dV/dt** massima (salita) e minima (discesa).

Le correnti sono registrate direttamente dal multimetro (sono `recordable` nel modello).
Segno: **positivo = corrente entrante** (depolarizza), negativo = uscente (ripolarizza).

In [ ]:
STEP_AMP = 300.0     # pA, ampiezza del gradino (Choudhury non la riporta: scelta sopra la soglia del modello)
T_ON, T_STEP = 100.0, 200.0
THR_DVDT = 50.0      # mV/ms, criterio di soglia. Non 10 mV/ms: all'inizio del gradino la membrana si carica
                     # gia' a circa STEP_AMP / C_m = 30 mV/ms, e con 10 la "soglia" cadrebbe sul riposo


def first_spike(t, V):
    # Misure sul primo spike dopo l'inizio del gradino. Restituisce None se non ci sono spike.
    rt = t - (T_ON + dt)
    rest = V[(rt >= -10) & (rt < 0)].mean()
    on = (rt >= 0) & (rt <= T_STEP)
    idx = np.where(on)[0]
    above = idx[V[idx] > 0]
    if len(above) == 0:
        return None
    # picco = massimo del primo tratto sopra 0 mV
    i0 = above[0]
    i_end = i0 + np.argmax(V[i0:] < 0)
    ip = i0 + np.argmax(V[i0:i_end])
    dvdt = np.gradient(V, t)
    # soglia: partendo dal punto di massima pendenza della salita si torna indietro
    # finche' dV/dt resta sopra THR_DVDT
    start = max(idx[0], i0 - int(5.0 / dt))
    j = start + np.argmax(dvdt[start:ip + 1])
    while j > start and dvdt[j - 1] > THR_DVDT:
        j -= 1
    thr = V[j]
    half = rest + (V[ip] - rest) / 2
    i1 = j + np.argmax(V[j:ip + 1] >= half)
    i2 = ip + np.argmax(V[ip:] <= half)
    # interpolazione lineare per una half-width piu' precisa del passo dt
    t1 = np.interp(half, V[i1 - 1:i1 + 1], t[i1 - 1:i1 + 1])
    t2 = np.interp(half, V[i2:i2 - 2:-1], t[i2:i2 - 2:-1])
    lo, hi = rest + 0.1 * (V[ip] - rest), rest + 0.9 * (V[ip] - rest)
    r1 = j + np.argmax(V[j:ip + 1] >= lo)
    r2 = j + np.argmax(V[j:ip + 1] >= hi)
    w_ahp = (t > t[ip]) & (t <= t[ip] + 5)
    return dict(i_peak=ip, t_peak=t[ip], rest=rest, peak=V[ip], amplitude=V[ip] - rest, threshold=thr,
                amp_from_thr=V[ip] - thr,
                half_width=t2 - t1, rise_10_90=t[r2] - t[r1], ahp=V[w_ahp].min() - thr,
                dvdt_max=dvdt[i1 - 50:ip].max(), dvdt_min=dvdt[ip:i2 + 50].min())


C1 = {}
for name, p in GENOTYPES.items():
    t, rec = simulate(p, [T_ON, T_ON + T_STEP], [STEP_AMP, 0.0], T_ON + T_STEP + 50.0,
                      record=["V_m"] + CURRENTS)
    C1[name] = (t, rec, first_spike(t, rec["V_m"]))

print(f"Gradino di {STEP_AMP:g} pA per {T_STEP:g} ms")
print(f"{'genotipo':<20}{'n spike':>8}{'picco':>8}{'ampiezza':>10}{'amp. da soglia':>15}{'soglia':>9}{'half-w':>9}{'salita':>9}{'AHP':>8}{'dV/dt max':>11}{'dV/dt min':>11}")
for name, (t, rec, s) in C1.items():
    V = rec["V_m"]
    w = (t >= T_ON) & (t <= T_ON + T_STEP)
    n_sp = int(np.sum((V[w][:-1] < 0) & (V[w][1:] >= 0)))
    if s is None:
        print(f"{name:<20}{n_sp:>8}   nessuno spike: aumentare STEP_AMP")
        continue
    print(f"{name:<20}{n_sp:>8}{s['peak']:>8.1f}{s['amplitude']:>10.1f}{s['amp_from_thr']:>15.1f}{s['threshold']:>9.1f} {s['half_width']:>8.3f} "
          f"{s['rise_10_90']:>8.3f} {s['ahp']:>7.1f} {s['dvdt_max']:>10.0f} {s['dvdt_min']:>10.0f}")
print("unita': mV, ms, mV/ms")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))

# (a) primo spike sovrapposto per i 4 genotipi, allineato al picco
for name, (t, rec, s) in C1.items():
    if s is None:
        continue
    ax[0, 0].plot(t - s["t_peak"], rec["V_m"], color=COLORS[name], label=f"{name}  (hw = {s['half_width']:.2f} ms)")
ax[0, 0].set_xlim(-1.5, 3); ax[0, 0].set_title("Primo spike, allineato al picco  [Choudhury Fig. 6A]")
ax[0, 0].set_xlabel("t dal picco (ms)"); ax[0, 0].set_ylabel("V (mV)"); ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=0.3)

# (b) half-width: modello vs Choudhury (e Markkula per il WT)
paper_hw = {"WT": (0.31, 0.08), "Kv3.1 KO": (0.43, 0.09), "Kv3.3 KO": (0.44, 0.09)}
names = list(GENOTYPES)
x = np.arange(len(names))
mod_hw = [C1[n][2]["half_width"] if C1[n][2] is not None else np.nan for n in names]
ax[0, 1].bar(x - 0.2, mod_hw, 0.4, color=[COLORS[n] for n in names], alpha=0.8, label="modello")
ax[0, 1].bar([x[i] + 0.2 for i, n in enumerate(names) if n in paper_hw], [paper_hw[n][0] for n in names if n in paper_hw],
             0.4, yerr=[paper_hw[n][1] for n in names if n in paper_hw], color="lightgrey", edgecolor="k",
             capsize=4, label="Choudhury 2020 (P14-P25)")
ax[0, 1].axhline(0.221, color="grey", ls="--", lw=1, label="Markkula WT (1 mese) 0.221 ms")
ax[0, 1].set_xticks(x); ax[0, 1].set_xticklabels(names, fontsize=8)
ax[0, 1].set_ylabel("half-width (ms)"); ax[0, 1].set_title("Half-width  [Choudhury Fig. 6B]")
ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=0.3, axis="y")

# (c) phase plot dV/dt vs V
for name, (t, rec, s) in C1.items():
    if s is None:
        continue
    w = (t >= s["t_peak"] - 1.5) & (t <= s["t_peak"] + 3)
    ax[0, 2].plot(rec["V_m"][w], np.gradient(rec["V_m"], t)[w], color=COLORS[name], label=name)
ax[0, 2].set_title("Phase plot del primo spike"); ax[0, 2].set_xlabel("V (mV)"); ax[0, 2].set_ylabel("dV/dt (mV/ms)")
ax[0, 2].legend(fontsize=8); ax[0, 2].grid(alpha=0.3)

# (d)-(f) correnti durante il primo spike: WT, Kv3.1 KO, Kv3.3 KO
cur_col = {"I_Na": "tab:orange", "I_LT": "tab:purple", "I_Kv3_1": "tab:green", "I_Kv3_3": "tab:red", "I_L": "grey"}
for k, name in enumerate(["WT", "Kv3.1 KO", "Kv3.3 KO"]):
    t, rec, s = C1[name]
    a = ax[1, k]
    if s is None:
        continue
    for c in CURRENTS:
        a.plot(t - s["t_peak"], rec[c] / 1000.0, color=cur_col[c], label=c)
    a2 = a.twinx()
    a2.plot(t - s["t_peak"], rec["V_m"], color="k", lw=0.8, ls=":")
    a2.set_ylabel("V (mV), linea puntinata", fontsize=8)
    a.set_xlim(-1.5, 3); a.set_ylim(-6, 6)
    a.set_title(f"Correnti nel primo spike: {name}"); a.set_xlabel("t dal picco (ms)")
    a.set_ylabel("I (nA)   + entrante / − uscente"); a.grid(alpha=0.3); a.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    s = C1[name][2]
    if s is not None:
        summary[f"C1 half-width {name} (ms)"] = (s["half_width"], {"WT": "0.31 ± 0.08", "Kv3.1 KO": "0.43 ± 0.09", "Kv3.3 KO": "0.44 ± 0.09"}[name], "Fig. 6B")
s = C1["WT"][2]
if s is not None:
    summary["C1 ampiezza WT (mV)"] = (s["amplitude"], "61.1 ± 7.9", "da −60 mV (testo)")
    summary["C1 soglia WT (mV)"] = (s["threshold"], "−38.2 ± 6.7", "testo")
    summary["C1 AHP WT (mV)"] = (s["ahp"], "−10.9 ± 2.5", "testo")
    summary["C1 salita 10-90% WT (ms)"] = (s["rise_10_90"], "0.136 ± 0.017", "testo")

## C2 – Voltage clamp: curva I/V e TEA (Choudhury Fig. 1 e Fig. 7)

**Come funziona il voltage clamp qui.** In voltage clamp la tensione è imposta, quindi l'equazione di
membrana sparisce e restano solo i gate. Con V costante a tratti ogni gate ha soluzione esatta
$x(t) = x_\infty + (x_0 - x_\infty)\,e^{-t/\tau_x}$ (come nel notebook `expvoltageclamp`), quindi non serve NEST.
I parametri dei gate sono letti da `nest.GetDefaults` del modello compilato.

**Protocollo (Fig. 1C):** holding −60 mV, prepulse a −90 mV per 200 ms (toglie l'inattivazione), poi −40 mV
per 20 ms (inattiva il sodio), poi gradini da −90 a +40 mV ogni 10 mV. La durata del gradino non è nel
testo: `T_STEP_VC` è una stima. Si misura la corrente di **picco** durante il gradino.

- Il **sodio è escluso** (`INCLUDE_NA = False`): nel paper è inattivato dal prepulse a −40 mV.
- Il **leak è incluso** (`INCLUDE_LEAK = True`): le I/V di Fig. 7 sono "without leak subtraction".
- **TEA 1 mM** blocca la frazione `TEA_BLOCK` di entrambe le correnti Kv3.

Segno: qui **positivo = corrente uscente**, come nelle figure di voltage clamp.

In [ ]:
INCLUDE_NA = False
INCLUDE_LEAK = True
TEA_BLOCK = 1.0          # frazione di Kv3 bloccata da TEA 1 mM
T_STEP_VC = 100.0        # ms, durata del gradino (stima)
STEPS_VC = np.arange(-90, 41, 10)
GATES = ["m", "h", "l", "r", "n1", "p1", "n3", "p3"]


def rates(P, g, V):
    a = P["k_alpha_" + g] * np.exp(P["eta_alpha_" + g] * V)
    b = P["k_beta_" + g] * np.exp(P["eta_beta_" + g] * V)
    return a, b


def x_inf(P, g, V):
    a, b = rates(P, g, V)
    return a / (a + b)


def tau_x(P, g, V):
    a, b = rates(P, g, V)
    return 1.0 / (a + b)


def vc_currents(P, s, V, tea=0.0):
    # Correnti in nA, positive = uscenti: g * (V - E)
    I = {
        "Kv3.1": (1 - tea) * P["g_Kv3_1"] * s["n1"]**3 * (0.8 + 0.2 * s["p1"]) * (V - P["E_K"]) / 1000.0,
        "Kv3.3": (1 - tea) * P["g_Kv3_3"] * s["n3"]**3 * (0.8 + 0.2 * s["p3"]) * (V - P["E_K"]) / 1000.0,
        "LT": P["g_LT"] * s["l"] * s["r"] * (V - P["E_K"]) / 1000.0,
        "leak": (P["g_L"] * (V - P["E_L"]) / 1000.0) * (1.0 if INCLUDE_LEAK else 0.0),
        "Na": (P["g_Na"] * s["m"]**3 * s["h"] * (V - P["E_Na"]) / 1000.0) * (1.0 if INCLUDE_NA else 0.0),
    }
    I["total"] = I["Kv3.1"] + I["Kv3.3"] + I["LT"] + I["leak"] + I["Na"]
    return I


def clamp(P, protocol, tea=0.0, step=0.01):
    # protocol: lista di (V, durata). I gate partono allo stato stazionario del primo tratto (holding).
    state = {g: x_inf(P, g, protocol[0][0]) for g in GATES}
    T, VV, G = [], [], {g: [] for g in GATES}
    t0 = 0.0
    for V, dur in protocol:
        n = max(int(round(dur / step)), 1)
        tt = np.arange(n) * step
        for g in GATES:
            xi, tau = x_inf(P, g, V), tau_x(P, g, V)
            G[g].append(xi + (state[g] - xi) * np.exp(-tt / tau))
            state[g] = xi + (state[g] - xi) * np.exp(-n * step / tau)
        T.append(t0 + tt); VV.append(np.full(n, float(V)))
        t0 += n * step
    t = np.concatenate(T); Vt = np.concatenate(VV)
    s = {g: np.concatenate(G[g]) for g in GATES}
    return t, Vt, vc_currents(P, s, Vt, tea)


def iv_family(P, prepulses, hold=-60.0, tea=0.0, steps=STEPS_VC):
    # Per ogni gradino: holding (10 ms) -> prepulses -> gradino -> holding (20 ms).
    # Restituisce tracce (t riferito all'inizio del gradino) e corrente di picco durante il gradino.
    fam, peak = {}, []
    t_pre = 10.0 + sum(d for _, d in prepulses)
    for Vs in steps:
        prot = [(hold, 10.0)] + list(prepulses) + [(Vs, T_STEP_VC), (hold, 20.0)]
        t, Vt, I = clamp(P, prot, tea)
        tr = t - t_pre
        fam[Vs] = (tr, I)
        w = (tr >= 0) & (tr < T_STEP_VC)
        peak.append(I["total"][w][np.argmax(np.abs(I["total"][w]))])
    return fam, np.array(peak)


def P_of(genotype):
    return {**DEFAULTS, **GENOTYPES[genotype]}


PRE_C2 = [(-90.0, 200.0), (-40.0, 20.0)]
C2 = {}
for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    C2[name] = iv_family(P_of(name), PRE_C2)
    C2[name + " + TEA"] = iv_family(P_of(name), PRE_C2, tea=TEA_BLOCK)
i10 = int(np.where(STEPS_VC == 10)[0][0])
for k, (fam, pk) in C2.items():
    print(f"{k:<20} picco a +10 mV = {pk[i10]:6.2f} nA")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
for k, name in enumerate(["WT", "Kv3.1 KO", "Kv3.3 KO"]):
    fam, _ = C2[name]
    for Vs, (tr, I) in fam.items():
        ax[0, k].plot(tr, I["total"], lw=0.9, color=plt.cm.viridis((Vs + 90) / 130))
    ax[0, k].set_xlim(-5, T_STEP_VC + 10)
    ax[0, k].set_title(f"{name}: gradini −90…+40 mV  [Choudhury Fig. 7A]")
    ax[0, k].set_xlabel("t dall'inizio del gradino (ms)"); ax[0, k].set_ylabel("I (nA), + uscente"); ax[0, k].grid(alpha=0.3)
for k, name in enumerate(["Kv3.1 KO", "Kv3.3 KO"]):
    a = ax[1, k]
    a.plot(STEPS_VC, C2["WT"][1], "o-", color="grey", label="WT")
    a.plot(STEPS_VC, C2[name][1], "o-", color=COLORS[name], label=name)
    a.plot(STEPS_VC, C2[name + " + TEA"][1], "s--", color=COLORS[name], mfc="w", label=name + " + TEA")
    a.set_title(f"I/V di picco: WT vs {name}  [Fig. 7{'B' if k == 0 else 'C'}]")
    a.set_xlabel("V (mV)"); a.set_ylabel("I picco (nA)"); a.grid(alpha=0.3); a.legend(fontsize=8)
# bar graph a +10 mV (Fig. 7D)
labels = ["WT", "Kv3.1 KO", "Kv3.1 KO + TEA", "Kv3.3 KO", "Kv3.3 KO + TEA"]
paper = {"WT": (11.9, 1.2), "Kv3.1 KO": (9.8, 2.8), "Kv3.1 KO + TEA": (6.1, 1.4), "Kv3.3 KO": (10.1, 3.0), "Kv3.3 KO + TEA": (5.6, 2.0)}
x = np.arange(len(labels))
ax[1, 2].bar(x - 0.2, [C2[l][1][i10] for l in labels], 0.4, color="tab:blue", alpha=0.8, label="modello")
ax[1, 2].bar(x + 0.2, [paper[l][0] for l in labels], 0.4, yerr=[paper[l][1] for l in labels],
             color="lightgrey", edgecolor="k", capsize=4, label="Choudhury 2020")
ax[1, 2].set_xticks(x); ax[1, 2].set_xticklabels(labels, rotation=20, fontsize=8)
ax[1, 2].set_ylabel("I picco a +10 mV (nA)"); ax[1, 2].set_title("Corrente a +10 mV  [Fig. 7D]")
ax[1, 2].legend(fontsize=8); ax[1, 2].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

# composizione della corrente WT a +10 mV, per capire chi la porta
tr, I = C2["WT"][0][10]
w = (tr >= 0) & (tr < T_STEP_VC)
ip = np.argmax(I["total"][w])
print("WT a +10 mV, al picco:  " + ",  ".join(f"{c} = {I[c][w][ip]:.2f} nA" for c in ["Kv3.1", "Kv3.3", "LT", "leak"]))

for l in labels:
    summary[f"C2 I(+10 mV) {l} (nA)"] = (C2[l][1][i10], f"{paper[l][0]} ± {paper[l][1]}", "Fig. 7D")

## C3 – Holding −60 contro −40 mV: inattivazione di Kv3 (Choudhury Fig. 8)

Choudhury confronta le I/V partendo da un holding di −60 mV e di −40 mV, per vedere se tenere la cellula
depolarizzata (come durante treni sinaptici) inattiva i canali Kv3. Il protocollo esatto è solo nella figura:
qui la cellula è tenuta a lungo all'holding (gate allo stato stazionario) e poi si applicano i gradini,
**senza prepulse**.

In [ ]:
C3 = {}
for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    for hp in (-60.0, -40.0):
        C3[(name, hp)] = iv_family(P_of(name), [], hold=hp)[1]

paper_c3 = {("WT", -60.0): (9.59, 2.37), ("WT", -40.0): (8.53, 1.87),
            ("Kv3.1 KO", -60.0): (6.19, 1.74), ("Kv3.1 KO", -40.0): (4.45, 1.11),
            ("Kv3.3 KO", -60.0): (7.58, 2.19), ("Kv3.3 KO", -40.0): (5.58, 1.24)}

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    ax[0].plot(STEPS_VC, C3[(name, -60.0)], "o-", color=COLORS[name], label=f"{name}, HP −60")
    ax[0].plot(STEPS_VC, C3[(name, -40.0)], "s--", color=COLORS[name], mfc="w", label=f"{name}, HP −40")
ax[0].set_title("I/V di picco da HP −60 e −40 mV  [Choudhury Fig. 8]"); ax[0].set_xlabel("V (mV)")
ax[0].set_ylabel("I picco (nA), + uscente"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=7)
keys = list(paper_c3)
x = np.arange(len(keys))
ax[1].bar(x - 0.2, [C3[k][i10] for k in keys], 0.4, color="tab:blue", alpha=0.8, label="modello")
ax[1].bar(x + 0.2, [paper_c3[k][0] for k in keys], 0.4, yerr=[paper_c3[k][1] for k in keys],
          color="lightgrey", edgecolor="k", capsize=4, label="Choudhury 2020")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{k[0]}\nHP {k[1]:.0f}" for k in keys], fontsize=8)
ax[1].set_ylabel("I picco a +10 mV (nA)"); ax[1].set_title("Corrente a +10 mV  [Fig. 8D]")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

for k in keys:
    summary[f"C3 I(+10) {k[0]} HP {k[1]:.0f} (nA)"] = (C3[k][i10], f"{paper_c3[k][0]} ± {paper_c3[k][1]}", "Fig. 8D")

## C4 – Treni ad alta frequenza: curve stimolo-risposta (Choudhury Fig. 10)

Impulsi di corrente da 0.3 ms per 500 ms a 100, 200, 300 e 400 Hz, con ampiezza da 0.5 a 3.5 nA.
Si conta il numero di spike evocati (picco sopra −10 mV entro un periodo dall'impulso, come negli altri
notebook). Nel paper, nell'MNTB tutti e tre i genotipi seguono fino a 400 Hz (Fig. 10C).

Questo esperimento è il più lento del notebook (circa un minuto).

In [ ]:
FREQS = [100, 200, 300, 400]
AMPS = np.arange(500.0, 3501.0, 500.0)    # pA
T_TRAIN, PULSE, START = 500.0, 0.3, 50.0


def train_count(params, freq, amp):
    per = 1000.0 / freq
    n_p = int(round(T_TRAIN / per))
    at, av, on = [], [], []
    for i in range(n_p):
        t0 = round((START + i * per) / dt) * dt
        at += [t0, round((START + i * per + PULSE) / dt) * dt]
        av += [amp, 0.0]
        on.append(t0 + dt)
    t, rec = simulate(params, at, av, round((START + n_p * per + 10) / dt) * dt)
    V = rec["V_m"]
    peaks = np.array([V[(t >= t0) & (t < t0 + per)].max() for t0 in on])
    return int((peaks >= -10).sum()), n_p, t, V


C4 = {}
for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    for f in FREQS:
        C4[(name, f)] = [train_count(GENOTYPES[name], f, a)[0] for a in AMPS]

fig, ax = plt.subplots(2, 4, figsize=(18, 8))
for j, f in enumerate(FREQS):
    n_max = int(round(T_TRAIN * f / 1000.0))
    for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
        ax[0, j].plot(AMPS / 1000.0, C4[(name, f)], "o-", color=COLORS[name], label=name)
    ax[0, j].axhline(n_max, color="k", ls=":", lw=1)
    ax[0, j].set_title(f"{f} Hz  [Choudhury Fig. 10B]"); ax[0, j].set_xlabel("ampiezza impulso (nA)")
    ax[0, j].set_ylabel("n spike evocati"); ax[0, j].grid(alpha=0.3)
ax[0, 0].legend(fontsize=8)

# esempio di traccia a 400 Hz con l'ampiezza massima (Fig. 10A)
for j, name in enumerate(["WT", "Kv3.1 KO", "Kv3.3 KO", "TEA (Kv3 bloccato)"]):
    n_ok, n_p, t, V = train_count(GENOTYPES[name], 400, AMPS[-1])
    ax[1, j].plot(t, V, color=COLORS[name], lw=0.7)
    ax[1, j].set_xlim(START - 5, START + 60)
    ax[1, j].set_title(f"{name}: 400 Hz, {AMPS[-1] / 1000:g} nA -> {n_ok}/{n_p}  [Fig. 10A]", fontsize=9)
    ax[1, j].set_xlabel("t (ms)"); ax[1, j].set_ylabel("V (mV)"); ax[1, j].grid(alpha=0.3)
plt.tight_layout(); plt.show()

for name in ["WT", "Kv3.1 KO", "Kv3.3 KO"]:
    summary[f"C4 400 Hz, 3.5 nA {name} (% spike)"] = (100.0 * C4[(name, 400)][-1] / 200, "≈ 100", "Fig. 10C")

## Riepilogo: modello vs Choudhury 2020

In [ ]:
print(f"{'misura':<40}{'modello':>10}   {'Choudhury 2020':<18}note")
print("-" * 85)
for k, (mod, ref, note) in summary.items():
    print(f"{k:<40}{mod:>10.3f}   {ref:<18}{note}")

## Come leggere i risultati

- **Con la divisione 50/50 e cinetiche identiche** Kv3.1 KO e Kv3.3 KO sono per forza uguali tra loro:
  ognuno toglie metà di Kv3. È coerente con Choudhury nell'MNTB (i due KO hanno la stessa half-width),
  ma non dice ancora nulla su differenze tra le subunità.
- **C1:** il confronto chiave è l'**aumento** di half-width dal WT ai KO (Choudhury: da 0.31 a circa 0.44 ms,
  cioè ×1.4). Se nel modello l'aumento è molto più piccolo, Kv3 conta poco nella ripolarizzazione (lo avevamo
  già visto: buona parte la fa I_LT).
- **C2:** Choudhury vede che nei KO la corrente totale **non cambia** (la subunità rimasta compensa), mentre
  nel modello togliere metà di g_Kv3 la riduce. Se si vuole riprodurre la compensazione, nel KO la
  conduttanza rimasta andrebbe aumentata (per esempio g_Kv3_3 = 150 nS nel Kv3.1 KO).
  Il valore assoluto (circa 12 nA nel WT) dice anche se la g_Kv3 totale del modello è realistica:
  la taglia della cellula (10 pF contro 11 pF) è quasi uguale.
- **C3:** se la corrente da HP −40 è quasi uguale a quella da HP −60, Kv3 si inattiva poco, come nel paper.
- **C4:** nell'MNTB vero tutti i genotipi seguono fino a 400 Hz.
- **Per i prossimi passi** (cambiare parametri): Choudhury riporta in letteratura una metà-attivazione di
  +6…+18 mV per Kv3.1 e +8…+30 mV per Kv3.3, cioè Kv3.3 si apre a tensioni un po' più alte. Si può provare
  a spostare `eta_alpha_n3` / `k_alpha_n3` per dare a Kv3.3 una cinetica diversa da Kv3.1.